# 01 Assemble the two tables

I build two tables from the ONS's own workbooks and keep them as two files. They sit on different
question sets and different age bands, and I don't want any later step to join them by accident:

1. `table1_new_questions` uses the new domestic abuse questions, ages 16 and over, for YE March 2024 to YE March 2026.
   YE March 2026 is the first full-sample year and the only one with published confidence intervals.
2. `table2_old_questions` uses the questions in use from April 2004. The long-run series is ages 16 to 59 (YE March 2005
   to YE March 2024). A short 16+ run (YE March 2022 to 2024) is included for reference only.

Reference period, questions basis, sample design and age band are explicit columns in both tables.

**What these figures are.** Each estimate is the share of people aged 16 and over (or 16 to 59) in
England and Wales who told the Crime Survey for England and Wales they had experienced domestic abuse
in the previous 12 months. They describe millions of people's experience, and I treat them that way.

**The ONS's assessment, which I hold every table and label to.** The year ending March 2026 increase
*includes real change*. Changes to the survey at the start of that year likely affected its size, by an
amount that cannot yet be measured. Nothing in this project estimates that amount, and nothing here
should be read as suggesting the increase is only a survey effect.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from common import (PAGES, PAGE_SOURCES, POPULATION, PROCESSED, RAW, RETRIEVED, SOURCES, TABLES,
                    page_text, population_16_plus, read_sheet, row, sha256)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
PROCESSED.mkdir(parents=True, exist_ok=True)

## Source manifest

I record every raw file and page snapshot with its size and SHA-256, so anyone can tell whether the ONS
still serves the bytes I verified against on the retrieval date.

In [ ]:
manifest = pd.DataFrame(
    [{"path": f"data/raw/{rel}", "kind": "table", "source_url": url, "retrieved": RETRIEVED,
      "bytes": (RAW / rel).stat().st_size, "sha256": sha256(RAW / rel)} for rel, url in SOURCES.items()]
    + [{"path": f"data/raw/source_pages/{RETRIEVED}/{name}", "kind": "page snapshot", "source_url": url,
        "retrieved": RETRIEVED, "bytes": (PAGES / name).stat().st_size, "sha256": sha256(PAGES / name)}
       for name, url in PAGE_SOURCES.items()]
)
manifest.to_parquet(PROCESSED / "source_manifest.parquet", index=False)
manifest.to_csv(PROCESSED / "source_manifest.csv", index=False)
manifest.assign(sha256=manifest.sha256.str[:16])[["path", "kind", "bytes", "sha256"]]

## Table 1: new questions, ages 16 and over

From the *Crime in England and Wales: Annual supplementary tables*, YE March 2026 edition (domestic abuse
sheets added 22 October 2026):

* **A9a**: YE March 2026 estimates with 95% confidence intervals.
* **A9c**: YE March 2024, 2025 and 2026 on the new questions, with the ONS's significance test of 2026 against 2025.

YE March 2024 and 2025 come from the split-sample trial, where about half of respondents answered the new
questions. From April 2025 everyone did, on new survey software, with the self-completion section on a tablet
rather than a laptop. I record that in the `sample_design` and `survey_implementation` columns rather than
smoothing over it.

In [ ]:
AST = TABLES / "annualsupplementarytables_yemar2026.xlsx"
a9a = read_sheet(AST, "Table A9a", "Type of abuse")
a9c = read_sheet(AST, "Table A9c", "Type of abuse")

# My short name for each measure -> the row label the ONS uses in A9a and A9c.
MEASURES = {"any domestic abuse": "Any domestic abuse",
            "any partner abuse": "Any partner abuse",
            "any family abuse": "Any family abuse"}

# One entry per survey year on the new questions. The two set-ups are described in the ONS blog and bulletin.
NEW_YEARS = [
    ("Apr 2023 to Mar 2024", "2024-03", "split sample: about half of respondents",
     "set-up before April 2025"),
    ("Apr 2024 to Mar 2025", "2025-03", "split sample: about half of respondents",
     "set-up before April 2025"),
    ("Apr 2025 to Mar 2026", "2026-03", "full sample",
     "from April 2025: new survey software; self-completion on tablet (was laptop); minor wording changes; "
     "relationship-status question set-up corrected"),
]


def col(df, startswith, endswith=""):
    hits = [c for c in df.columns if c.startswith(startswith) and c.endswith(endswith)]
    if len(hits) != 1:
        raise ValueError(f"expected one column starting {startswith!r}, found {hits}")
    return hits[0]


bases = row(a9c, "Unweighted base")
sig_col = col(a9c, "Apr 2025 to Mar 2026 compared with", "All")
records = []
for period, year_ending, sample, implementation in NEW_YEARS:
    est_col = col(a9c, f"{period} All")
    for measure, label in MEASURES.items():
        r = row(a9c, label)
        rec = dict(reference_period=period, year_ending=year_ending,
                   questions_basis="new questions (first asked April 2023)",
                   sample_design=sample, survey_implementation=implementation,
                   age_band="16 and over", sex="All", measure=measure,
                   estimate_pct=float(r[est_col]), ci_lower_pct=np.nan, ci_upper_pct=np.nan,
                   ci_published=False, change_vs_previous_year=None,
                   unweighted_base=int(bases[est_col]),
                   source_file=AST.name, source_table="A9c")
        if year_ending == "2026-03":
            # For YE March 2026 I take the estimate and its interval from A9a, where the interval is published.
            ra = row(a9a, label)
            rec.update(estimate_pct=float(ra["All in the last year Estimate"]),
                       ci_lower_pct=float(ra["All in the last year Confidence interval Lower bound"]),
                       ci_upper_pct=float(ra["All in the last year Confidence interval Upper bound"]),
                       ci_published=True, change_vs_previous_year=r[sig_col], source_table="A9a (CI), A9c (test)")
        records.append(rec)

table1 = pd.DataFrame(records)
table1.round(2)

## Table 2: old questions

From *Domestic abuse prevalence and victim characteristics*, YE March 2024 edition. That's the last edition to
carry the old-questions series. The YE March 2025 edition moved to the new questions only.

* **Table 4** is the long-run series for ages 16 to 59. The ONS keeps this age band because the self-completion
  section had an upper age limit of 59 until October 2021.
* **Table 1b** is the old questions for ages 16 and over, YE March 2022 to 2024. I keep it for reference only.
  It is not part of the long run, and it is not comparable with Table 1. The new questions count a person as a
  victim from one abusive behaviour, in line with the Domestic Abuse Act 2021 definition, and cover more types
  of abuse. The ONS states the two sets of estimates are not comparable.

The long run has gaps of its own, and I keep them as rows with no estimate rather than letting a line run across
them: YE March 2008 (comparable questions not asked) and YE March 2021 (domestic abuse questions excluded from the
telephone survey during the pandemic).

In [ ]:
DAPVC24 = TABLES / "domesticabuseprevalenceandvictimcharacteristics_yemar2024.xlsx"
t4 = read_sheet(DAPVC24, "Table 4", "Type of abuse")
t1b = read_sheet(DAPVC24, "Table 1b", "Type of abuse")

OLD_MEASURES = {"any domestic abuse": "Any domestic abuse (",
                "any partner abuse": "Any partner abuse (",
                "any family abuse": "Any family abuse ("}

# Collection notes by year ending, paraphrasing the ONS notes to Table 4 (note numbers in brackets).
ADJUSTED = ("calculated from the original questions with an adjustment to make them comparable with the "
            "questions used from YE March 2013 (note 3)")
OLD_NOTES = {
    "2005-03": ADJUSTED + "; a manual adjustment was also applied to this year (note 5)",
    **{f"{y}-03": ADJUSTED for y in (2006, 2007, 2009, 2010)},
    **{f"{y}-03": ADJUSTED + "; smaller sample, split-sample experiment (note 7)" for y in (2011, 2012)},
    "2013-03": "smaller sample, split-sample experiment (note 7)",
    "2018-03": "smaller sample, split-sample experiment (note 8)",
    "2019-03": "smaller sample, split-sample experiment (note 8)",
    "2020-03": "face-to-face survey suspended 17 March 2020 (note 9)",
    "2022-03": "six months of interviews, October 2021 to March 2022 (note 10)",
    "2023-03": "eight months of interviews; October 2022 to January 2023 excluded after a survey error (note 11)",
    "2024-03": "old questions asked of about half of respondents during the split-sample trial of the new questions",
}
GAPS = {
    "2008-03": ("Apr 2007 to Mar 2008", "not available: comparable questions were not asked this year (note 6)"),
    "2021-03": ("Apr 2020 to Mar 2021", "not collected: domestic abuse questions were excluded from the telephone "
                                        "survey that replaced face-to-face interviews during the pandemic (note 9)"),
}


def period_columns(df):
    out = {}
    for c in df.columns:
        if " compared with " in c:
            continue
        parts = c.split(" ")
        if len(parts) >= 5 and parts[0] in ("Apr", "Oct") and parts[2] == "to" and parts[3] == "Mar":
            out[c] = (" ".join(parts[:5]), f"{parts[4]}-03")
    return out


def old_rows(df, table, age_band, long_run, sex_suffix=""):
    recs = []
    base = row(df, "Unweighted base")
    for c, (period, year_ending) in period_columns(df).items():
        if sex_suffix and not c.split(" [")[0].endswith(sex_suffix):
            continue
        for measure, label in OLD_MEASURES.items():
            v = row(df, label)[c]
            published = isinstance(v, (int, float))
            recs.append(dict(
                reference_period=period, year_ending=year_ending,
                questions_basis="old questions (in use from April 2004)",
                age_band=age_band, sex="All", measure=measure, long_run_series=long_run,
                estimate_pct=float(v) if published else np.nan,
                status="published" if published else GAPS[year_ending][1],
                collection_note=OLD_NOTES.get(year_ending, ""),
                unweighted_base=int(base[c]) if published else pd.NA,
                source_file=DAPVC24.name, source_table=table))
    return recs


recs = old_rows(t4, "Table 4", "16 to 59", True) + old_rows(t1b, "Table 1b", "16 and over", False, " All")

# YE March 2021 has no column in Table 4 at all, so I add it explicitly as a gap row.
for measure in OLD_MEASURES:
    recs.append(dict(reference_period=GAPS["2021-03"][0], year_ending="2021-03",
                     questions_basis="old questions (in use from April 2004)", age_band="16 to 59", sex="All",
                     measure=measure, long_run_series=True, estimate_pct=np.nan, status=GAPS["2021-03"][1],
                     collection_note="", unweighted_base=pd.NA, source_file=DAPVC24.name,
                     source_table="Table 4 (no column; note 9)"))

table2 = (pd.DataFrame(recs)
          .astype({"unweighted_base": "Int64"})
          .sort_values(["long_run_series", "measure", "year_ending"], ascending=[False, True, True])
          .reset_index(drop=True))
table2[table2.measure.eq("any domestic abuse")].round(2)

## Structural checks before saving

The two tables must not share a questions basis, and the long run must be one age band. If either fails, I
stop here rather than write files a chart could read.

In [ ]:
assert set(table1.questions_basis).isdisjoint(table2.questions_basis), "a questions basis appears in both tables"
assert set(table1.age_band) == {"16 and over"}
assert set(table2.loc[table2.long_run_series, "age_band"]) == {"16 to 59"}
assert not table2.loc[table2.long_run_series].duplicated(["measure", "year_ending"]).any()
assert table2.loc[table2.estimate_pct.isna(), "year_ending"].unique().tolist() == ["2008-03", "2021-03"]

for name, df in [("table1_new_questions", table1), ("table2_old_questions", table2)]:
    df.to_parquet(PROCESSED / f"{name}.parquet", index=False)
    df.to_csv(PROCESSED / f"{name}.csv", index=False)
    print(f"wrote {name}: {len(df)} rows")